In [1]:
# ═══════════════════════════════════════════════════════════════
# Notebook  : 05c_case_comparison_regression
# Project   : XAI-Driven DSR Prediction Framework for Youth Debt Crisis
# Purpose   : Case-level detailed comparison tables for DiCE counterfactuals
#             Original vs CF1 feature value changes (absolute + percentage)
#             Cross-case summary statistics for policy interpretation
# Data      : dice_cf_summary_reg.csv, dice_cf_full_reg.csv,
#             dice_cf_diversity_reg.csv, dice_cf_full_diversity_reg.csv
# Author    :
# ═══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────
# Cell 1 | Import libraries
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
import joblib

warnings.filterwarnings('ignore')

print("Libraries loaded successfully")


# ─────────────────────────────────────────────
# Cell 2 | Load model & results
# ─────────────────────────────────────────────
lgbm = joblib.load('../outputs/model_lgbm_reg.pkl')

# CF1 results (from 05_dice_counterfactual_regression)
cf_full_cf1    = pd.read_csv('../outputs/dice_cf_full_reg.csv')
cf_summary_cf1 = pd.read_csv('../outputs/dice_cf_summary_reg.csv')

# Diversity results (from 05b_dice_diversity_regression)
cf_full_div    = pd.read_csv('../outputs/dice_cf_full_diversity_reg.csv')
cf_summary_div = pd.read_csv('../outputs/dice_cf_diversity_reg.csv')

# Variable display name mapping
VAR_LABELS = {
    'sex'                : 'Sex',
    'education'          : 'Education level',
    'marital_status'     : 'Marital status',
    'region'             : 'Region',
    'employment_status'  : 'Employment status',
    'employment_type'    : 'Employment type',
    'h19_pers_income1'   : 'Regular wage income (10k KRW)',
    'h19_pers_income2'   : 'Temporary wage income (10k KRW)',
    'h19_pers_income4'   : 'Side job income (10k KRW)',
    'h19_cin'            : 'Household regular income (10k KRW)',
    'h19_din'            : 'Household disposable income (10k KRW)',
    'debt_financial'     : 'Financial institution loan (10k KRW)',
    'debt_private'       : 'Private loan (10k KRW)',
    'debt_card'          : 'Credit card debt (10k KRW)',
    'debt_lease'         : 'Lease deposit debt (10k KRW)',
    'debt_credit'        : 'Credit purchase debt (10k KRW)',
    'debt_other'         : 'Other debt (10k KRW)',
    'housing_debt_balance': 'Housing debt balance (10k KRW)',
    'housing_tenure'     : 'Housing tenure type',
    'age'                : 'Age',
    'total_debt'         : 'Total debt (10k KRW)',
}

feature_cols = [c for c in cf_full_cf1.columns
                if c not in ['type', 'case_id', 'cf_id', 'dsr_actual']]

print(f"CF1 full records     : {cf_full_cf1.shape}")
print(f"CF1 summary records  : {cf_summary_cf1.shape}")
print(f"Diversity full records: {cf_full_div.shape}")
print(f"Diversity summary    : {cf_summary_div.shape}")


# ─────────────────────────────────────────────
# Cell 3 | Build CF1 detailed comparison table
# Original vs CF1: feature values + absolute/pct change
# ─────────────────────────────────────────────
cf1_rows = []

for _, sum_row in cf_summary_cf1.iterrows():
    case_id   = sum_row['case_id']
    dsr_orig  = sum_row['dsr_actual']
    pred_orig = sum_row['dsr_pred_original']
    pred_cf1  = sum_row['dsr_pred_cf1']
    changed   = [c.strip() for c in str(sum_row['changed_features']).split(',')]

    sub      = cf_full_cf1[cf_full_cf1['case_id'] == case_id]
    orig_row = sub[sub['type'] == 'original']
    cf_rows_sub = sub[sub['type'] == 'CF1']

    if len(orig_row) == 0 or len(cf_rows_sub) == 0:
        continue

    orig = orig_row.iloc[0]
    cf1  = cf_rows_sub.iloc[0]

    for feat in changed:
        if feat not in feature_cols:
            continue
        try:
            orig_val   = float(orig[feat])
            cf_val     = float(cf1[feat])
            change_abs = cf_val - orig_val
            change_pct = (change_abs / orig_val * 100) if orig_val != 0 else np.nan
        except Exception:
            orig_val   = orig[feat]
            cf_val     = cf1[feat]
            change_abs = np.nan
            change_pct = np.nan

        cf1_rows.append({
            'Case ID'            : case_id,
            'DSR actual'         : round(dsr_orig, 4),
            'DSR pred (original)': round(pred_orig, 4),
            'DSR pred (CF1)'     : round(pred_cf1, 4),
            'DSR reduction'      : round(pred_orig - pred_cf1, 4),
            'Feature'            : VAR_LABELS.get(feat, feat),
            'Original value'     : round(orig_val, 2) if isinstance(orig_val, float) else orig_val,
            'CF1 value'          : round(cf_val, 2) if isinstance(cf_val, float) else cf_val,
            'Change (abs)'       : round(change_abs, 2) if not np.isnan(change_abs) else np.nan,
            'Change (%)'         : round(change_pct, 1) if not np.isnan(change_pct) else np.nan,
        })

cf1_detail = pd.DataFrame(cf1_rows)
print("=== CF1 Detailed Comparison Table ===")
print(cf1_detail.to_string(index=False))


# ─────────────────────────────────────────────
# Cell 4 | Build diversity comparison table
# CF1 vs CF2 vs CF3 per case: changed features side-by-side
# ─────────────────────────────────────────────
div_rows = []

for case_id in cf_summary_div['case_id'].unique():
    sub_sum = cf_summary_div[cf_summary_div['case_id'] == case_id]
    sub_full = cf_full_div[cf_full_div['case_id'] == case_id]

    orig_row = sub_full[sub_full['type'] == 'original']
    if len(orig_row) == 0:
        continue
    orig = orig_row.iloc[0]

    row = {
        'Case ID'       : case_id,
        'DSR actual'    : orig['dsr_actual'],
        'DSR pred orig' : None,
    }

    for cf_id in [1, 2, 3]:
        cf_sum = sub_sum[sub_sum['cf_id'] == cf_id]
        cf_full_row = sub_full[sub_full['type'] == f'CF{cf_id}']

        if len(cf_sum) == 0 or len(cf_full_row) == 0:
            row[f'CF{cf_id} pred DSR']    = np.nan
            row[f'CF{cf_id} n_changes']   = np.nan
            row[f'CF{cf_id} features']    = ''
            continue

        cs  = cf_sum.iloc[0]
        cfr = cf_full_row.iloc[0]

        if row['DSR pred orig'] is None:
            row['DSR pred orig'] = round(cs['dsr_pred_orig'], 4)

        changed = [c.strip() for c in str(cs['changed_features']).split(',')]
        changed_labels = [VAR_LABELS.get(f, f) for f in changed]

        row[f'CF{cf_id} pred DSR']  = round(cs['dsr_pred_cf'], 4)
        row[f'CF{cf_id} n_changes'] = int(cs['n_changes'])
        row[f'CF{cf_id} features']  = ' | '.join(changed_labels)

    div_rows.append(row)

div_detail = pd.DataFrame(div_rows)
print("\n=== Diversity Comparison Table (CF1 vs CF2 vs CF3) ===")
print(div_detail.to_string(index=False))


# ─────────────────────────────────────────────
# Cell 5 | Save all tables to CSV and Excel
# ─────────────────────────────────────────────
cf1_detail.to_csv('../outputs/cf_comparison_cf1_reg.csv',
                  index=False, encoding='utf-8-sig')
div_detail.to_csv('../outputs/cf_comparison_diversity_reg.csv',
                  index=False, encoding='utf-8-sig')

print("Saved -> outputs/cf_comparison_cf1_reg.csv")
print("Saved -> outputs/cf_comparison_diversity_reg.csv")

with pd.ExcelWriter('../outputs/cf_comparison_tables_reg.xlsx',
                    engine='openpyxl') as writer:
    cf1_detail.to_excel(writer, sheet_name='CF1_Detail', index=False)
    div_detail.to_excel(writer, sheet_name='CF_Diversity', index=False)
    cf_summary_cf1.to_excel(writer, sheet_name='CF1_Summary', index=False)
    cf_summary_div.to_excel(writer, sheet_name='Diversity_Summary', index=False)

print("Saved -> outputs/cf_comparison_tables_reg.xlsx")


# ─────────────────────────────────────────────
# Cell 6 | Cross-case feature consensus analysis
# Which features appear consistently across cases
# ─────────────────────────────────────────────
from collections import Counter

# CF1 feature frequency
cf1_changed_all = []
for _, row in cf_summary_cf1.iterrows():
    feats = [f.strip() for f in str(row['changed_features']).split(',')]
    cf1_changed_all.extend(feats)

cf1_freq = Counter(cf1_changed_all)

# All CFs feature frequency
div_changed_all = []
for _, row in cf_summary_div.iterrows():
    feats = [f.strip() for f in str(row['changed_features']).split(',')]
    div_changed_all.extend(feats)

div_freq = Counter(div_changed_all)

# Consensus table: features appearing in both CF1 and diversity
all_feats   = sorted(set(list(cf1_freq.keys()) + list(div_freq.keys())))
consensus_df = pd.DataFrame({
    'Feature'        : [VAR_LABELS.get(f, f) for f in all_feats],
    'CF1 frequency'  : [cf1_freq.get(f, 0) for f in all_feats],
    'All CFs frequency': [div_freq.get(f, 0) for f in all_feats],
}).sort_values('All CFs frequency', ascending=False).reset_index(drop=True)

consensus_df['Consensus'] = consensus_df.apply(
    lambda r: 'High' if r['CF1 frequency'] >= 3 and r['All CFs frequency'] >= 6
    else ('Medium' if r['CF1 frequency'] >= 2 or r['All CFs frequency'] >= 4
    else 'Low'), axis=1
)

print("\n=== Feature Consensus across CF1 and Diversity CFs ===")
print(consensus_df.to_string(index=False))

consensus_df.to_csv('../outputs/cf_feature_consensus_reg.csv',
                    index=False, encoding='utf-8-sig')
print("\nSaved -> outputs/cf_feature_consensus_reg.csv")


# ─────────────────────────────────────────────
# Cell 7 | Summary statistics
# ─────────────────────────────────────────────
print("\n=== Summary Statistics ===")

print("\n[CF1 - Primary Counterfactual]")
print(f"  Cases analyzed        : {cf_summary_cf1['case_id'].nunique()}")
print(f"  Mean DSR reduction    : {(cf_summary_cf1['dsr_pred_original'] - cf_summary_cf1['dsr_pred_cf1']).mean():.4f}")
print(f"  Max DSR reduction     : {(cf_summary_cf1['dsr_pred_original'] - cf_summary_cf1['dsr_pred_cf1']).max():.4f}")
print(f"  Mean features changed : {cf_summary_cf1['n_changes_cf1'].mean():.2f}")
top_feat_cf1 = cf1_freq.most_common(1)[0]
print(f"  Most changed feature  : {VAR_LABELS.get(top_feat_cf1[0], top_feat_cf1[0])} ({top_feat_cf1[1]}x)")

print("\n[Diversity CFs - CF1/CF2/CF3]")
print(f"  Cases analyzed        : {cf_summary_div['case_id'].nunique()}")
print(f"  Mean DSR reduction    : {(cf_summary_div['dsr_pred_orig'] - cf_summary_div['dsr_pred_cf']).mean():.4f}")
print(f"  Mean features changed : {cf_summary_div['n_changes'].mean():.2f}")
top_feat_div = div_freq.most_common(1)[0]
print(f"  Most changed feature  : {VAR_LABELS.get(top_feat_div[0], top_feat_div[0])} ({top_feat_div[1]}x)")

print("\n[High-consensus features (appearing in >= 3 CF1 cases)]")
high_consensus = [VAR_LABELS.get(f, f) for f, c in cf1_freq.most_common() if c >= 3]
for feat in high_consensus:
    print(f"  - {feat}")

Libraries loaded successfully
CF1 full records     : (40, 24)
CF1 summary records  : (10, 7)
Diversity full records: (40, 25)
Diversity summary    : (30, 8)
=== CF1 Detailed Comparison Table ===
 Case ID  DSR actual  DSR pred (original)  DSR pred (CF1)  DSR reduction                               Feature  Original value  CF1 value  Change (abs)  Change (%)
       1      0.6667               0.5490          0.1621         0.3869                     Employment status             2.0        6.5           4.5       225.0
       1      0.6667               0.5490          0.1621         0.3869         Regular wage income (10k KRW)          3769.0     8880.3        5111.3       135.6
       3      0.4125               1.5193          0.1831         1.3362       Temporary wage income (10k KRW)           480.0    12065.4       11585.4      2413.6
      12      0.5801               0.2946          0.0720         0.2226                  Other debt (10k KRW)             0.0     2684.5        2684